# Zusatzübung · Empfehlungen: beliebt oder zusammen gekauft?
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Jede E-Mail enthält bis zu drei Produktempfehlungen. Ihr vergleicht zwei Ansätze und beurteilt die Empfehlungen, ohne einen Kunden anzuschreiben.

**Kern:** Schritte 1 bis 4. **Optional:** E1, E2.

**Probiert es selbst.** Kommt ihr nicht weiter, fragt Gemini (✦-Knopf unten in der Mitte) oder einen anderen KI-Assistenten eurer Wahl.
Aber: Ihr müsst jede Zeile eurer Lösung verstehen und erklären können.

**Jeder Schritt:** Code in die leere Zelle › ▶ › **Plausibel?** prüfen. Die Lösung gibt es nach der Übung.

**Colab meldet «Zu viele Sitzungen»?** In den Notebooks der früheren Übungen: Laufzeit › «Verbindung trennen und Laufzeit löschen».

In [ ]:
#@title ▶ Setup: zuerst ausführen
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, sys, urllib.request
try:
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")   # immer die neuste Version
except OSError:
    if not os.path.exists("kurs.py"): raise
sys.modules.pop("kurs", None)   # neu laden, falls schon geladen
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
print("✅ Bereit.")

### ▶ BigQuery verbinden
Einfach ▶: `offline` ist voreingestellt, die Abfragen laufen über eine Kopie der Kursdaten. Das geht auch nach dem Kurstag.

*Am Kurstag* geht statt `offline` auch der Kurs-Schlüssel (BigQuery direkt), das Ergebnis ist dasselbe.

In [ ]:
#@title ▶ BigQuery verbinden
SCHLUESSEL = "offline"  #@param {type:"string"}
connect_bigquery(SCHLUESSEL, "fhnw-marketing-2026.paeckli")

### 1 Beliebt
**Ziel:** Die drei Produkte mit den meisten Kunden: allen dasselbe empfehlen.

**Aufgabe:** Welche 3 Produkte haben in `fhnw-marketing-2026.paeckli.orders_clean` die meisten verschiedenen Kunden gekauft? Zeige stock_code, description aus `fhnw-marketing-2026.paeckli.products` und die Anzahl Kunden.

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Echte Produkte, kein Porto?

### ▶ 2 Vorbereitet: «Zusammen gekauft» für jeden Kunden
Einfach ausführen. Die Zelle zählt, wie oft zwei Produkte auf derselben Rechnung stehen (die Tabelle von der Folie), und gibt jedem Kunden die 3 Produkte, die am häufigsten mit seinen Käufen zusammen gekauft werden.

In [ ]:
LETZTE = """letzte AS (       -- letzte Rechnung pro Kunde (mit mindestens 2 Rechnungen)
    SELECT customer_id, invoice FROM (
        SELECT customer_id, invoice,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY MAX(invoice_date) DESC, invoice DESC) AS r,
               COUNT(*) OVER (PARTITION BY customer_id) AS rechnungen
        FROM `fhnw-marketing-2026.paeckli.orders_clean` GROUP BY customer_id, invoice)
    WHERE r = 1 AND rechnungen >= 2)"""

def base_orders(training=False):
    """Die Bestellungen: alle, oder ohne die letzte Rechnung jedes Kunden (Training für Schritt 4)."""
    if training:
        return "WITH " + LETZTE + ",\nbasis AS (SELECT * FROM `fhnw-marketing-2026.paeckli.orders_clean` WHERE invoice NOT IN (SELECT invoice FROM letzte))"
    return "WITH basis AS (SELECT * FROM `fhnw-marketing-2026.paeckli.orders_clean`)"

def bought_together(training=False, ohne_gekaufte=False):
    filtern = """WHERE NOT EXISTS (SELECT 1 FROM gekauft schon       -- schon gekauft: nicht empfehlen
                         WHERE schon.customer_id = g.customer_id AND schon.stock_code = p.empfehlung)""" if ohne_gekaufte else ""
    return bq(f"""
    {base_orders(training)},
    rechnungen AS (SELECT DISTINCT invoice, stock_code FROM basis),
    paare AS (       -- auf wie vielen Rechnungen stehen zwei Produkte gemeinsam?
        SELECT a.stock_code AS produkt, b.stock_code AS empfehlung, COUNT(*) AS gemeinsam
        FROM rechnungen a JOIN rechnungen b ON a.invoice = b.invoice AND a.stock_code <> b.stock_code
        GROUP BY a.stock_code, b.stock_code),
    gekauft AS (SELECT DISTINCT customer_id, stock_code FROM basis),
    wertung AS (     -- Punkte pro Kunde und Produkt
        SELECT g.customer_id, p.empfehlung, SUM(p.gemeinsam) AS punkte
        FROM gekauft g JOIN paare p ON p.produkt = g.stock_code
        {filtern}
        GROUP BY g.customer_id, p.empfehlung),
    rang AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY punkte DESC, empfehlung) AS r FROM wertung)
    SELECT customer_id,
           MAX(CASE WHEN r = 1 THEN empfehlung END) AS rec_1,
           MAX(CASE WHEN r = 2 THEN empfehlung END) AS rec_2,
           MAX(CASE WHEN r = 3 THEN empfehlung END) AS rec_3
    FROM rang WHERE r <= 3
    GROUP BY customer_id
    """)

top3 = bq("""SELECT stock_code FROM `fhnw-marketing-2026.paeckli.orders_clean` GROUP BY stock_code
             ORDER BY COUNT(DISTINCT customer_id) DESC, stock_code LIMIT 3""")["stock_code"].tolist()   # die drei beliebtesten (Schritt 1)
alle_kunden = bq("SELECT DISTINCT customer_id FROM `fhnw-marketing-2026.paeckli.orders_clean` ORDER BY customer_id")
empfehlungen = alle_kunden.merge(bought_together(), on="customer_id", how="left")
for i, k in enumerate(["rec_1", "rec_2", "rec_3"]):   # wer keine Paare hat, bekommt die beliebtesten
    empfehlungen[k] = empfehlungen[k].fillna(top3[i])
empfehlungen.sample(5, random_state=1)

In [ ]:
sanity_check("empfehlungen", empfehlungen);

### 3 Passen die Empfehlungen?
**Ziel:** Selbst beurteilen: Würdet ihr bei diesen Empfehlungen klicken?

**Aufgabe:** Zeige für 3 zufällige Kunden aus empfehlungen (sample mit random_state=1), welche Produkte (description aus `fhnw-marketing-2026.paeckli.products`) sie in `fhnw-marketing-2026.paeckli.orders_clean` gekauft haben und welche drei Produkte ihnen empfohlen werden (rec_1, rec_2, rec_3).

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Passt das zum Gekauften? Bekommt jeder etwas anderes? Ist etwas dabei, das der Kunde schon hat?

### ▶ 4 Vorbereitet: Welcher Ansatz ist besser?
Einfach ausführen. Die Zelle versteckt die letzte Bestellung jedes Kunden, empfiehlt aus der Vorgeschichte und zählt, wie oft eine Empfehlung in der versteckten Bestellung vorkommt: die Trefferquote, für beide Ansätze.

In [ ]:
versteckt = bq(f"""
WITH {LETZTE}
SELECT DISTINCT l.customer_id, o.stock_code
FROM letzte l JOIN `fhnw-marketing-2026.paeckli.orders_clean` o ON o.invoice = l.invoice
""")

def hit_rate(recs):
    lang = recs.melt(id_vars="customer_id", value_vars=["rec_1", "rec_2", "rec_3"], value_name="stock_code")
    treffer = lang.merge(versteckt, on=["customer_id", "stock_code"])
    return treffer["customer_id"].nunique() / versteckt["customer_id"].nunique()

kunden = pd.DataFrame({"customer_id": versteckt["customer_id"].unique()})
top3_training = bq(f"""
{base_orders(training=True)}
SELECT stock_code FROM basis GROUP BY stock_code
ORDER BY COUNT(DISTINCT customer_id) DESC, stock_code LIMIT 3
""")["stock_code"].tolist()
beliebt_recs = kunden.assign(rec_1=top3_training[0], rec_2=top3_training[1], rec_3=top3_training[2])
zusammen_recs = kunden.merge(bought_together(training=True), on="customer_id", how="left")
vergleich = pd.Series({"Beliebt": hit_rate(beliebt_recs), "Zusammen gekauft": hit_rate(zusammen_recs)})
vergleich.plot.barh(title="Trefferquote: mindestens eine Empfehlung in der letzten Bestellung", figsize=(7, 2))
plt.show()
vergleich.round(3)

**Eure Einschätzung:** Ist «zusammen gekauft» klar besser? Und was heisst «besser» hier: ein Treffer in der nächsten Bestellung oder etwas, das der Kunde noch nicht kennt?

## Erweiterung (optional, für zu Hause)
Für alle, die Lust haben, nach dem Kurs weiterzumachen. **Im Kurs reicht die Zeit dafür nicht.**

### E1 Ausgelistete Produkte
**Ziel:** Produkte finden, die seit Juli niemand mehr kauft.

**Aufgabe:** Welche Produkte wurden in `fhnw-marketing-2026.paeckli.orders_clean` früher verkauft, aber seit dem 1. Juli 2025 nie mehr? Zeige stock_code, description (aus `fhnw-marketing-2026.paeckli.products`) und das letzte Verkaufsdatum. Wie oft kommen sie in empfehlungen vor?

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Werden sie empfohlen? Dann in `bought_together` herausfiltern.

### E2 Schon gekauft ausschliessen
**Ziel:** Nur Produkte empfehlen, die der Kunde noch nicht hat. Besser?

**Aufgabe:** Berechne die Trefferquote aus Schritt 4 nochmals, aber mit bought_together(training=True, ohne_gekaufte=True): Dann werden Produkte, die der Kunde schon gekauft hat, nicht empfohlen.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Sinkt die Trefferquote? Viele Kunden bestellen dasselbe wieder. Die Messmethode entscheidet, was «besser» heisst.